# Imports

In [1]:
import tensorflow as tf
tf.__version__
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2

# Load dataset

In [2]:
dataset_path = "dataset"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset ="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

Found 6652 files belonging to 6 classes.
Using 5322 files for training.
Found 6652 files belonging to 6 classes.
Using 1330 files for validation.


In [3]:
class_names = train_ds.class_names
print(class_names)

['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_healthy']


# Normalize Images

In [4]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y:(normalization_layer(x),y))
val_ds = val_ds.map(lambda x,y:(normalization_layer(x),y))

# Load Pretrained Model

We use MobileNetV2 as the feature extractor.

In [5]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
                         )
base_model.trainable = False

# Build the Model

In [6]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names),activation='softmax')
])

# Compile Model

In [7]:
model.compile(
    optimizer ='adam',
    loss ='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 57s 323ms/step - accuracy: 0.8435 - loss: 0.4301 - val_accuracy: 0.9150 - val_loss: 0.2185
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 51s 305ms/step - accuracy: 0.9365 - loss: 0.1888 - val_accuracy: 0.9271 - val_loss: 0.1950
Epoch 3/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 53s 316ms/step - accuracy: 0.9483 - loss: 0.1446 - val_accuracy: 0.9368 - val_loss: 0.1781
Epoch 4/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 53s 319ms/step - accuracy: 0.9630 - loss: 0.1089 - val_accuracy: 0.9504 - val_loss: 0.1305
Epoch 5/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 55s 332ms/step - accuracy: 0.9671 - loss: 0.0916 - val_accuracy: 0.9361 - val_loss: 0.1758


In [9]:

# model = tf.keras.models.load_model("model/plant_model.keras")

In [10]:
model.save("model/plant_model_fixed.keras")

In [11]:
model.export("model/plant_model_serving")

INFO:tensorflow:Assets written to: model/plant_model_serving\assets


INFO:tensorflow:Assets written to: model/plant_model_serving\assets


Saved artifact at 'model/plant_model_serving'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2052014445136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014447824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014446864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014447632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014444752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014444368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014448592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014448208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014444176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014443792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052014445904: Tensor

# test

In [12]:
import numpy as np
from PIL import Image
img = Image.open("test_image.jfif")
img = img.resize((224,224))

img_array = np.array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)

print(class_names[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 698ms/step
Tomato_Late_blight


In [13]:
import tensorflow as tf

model = tf.keras.models.load_model("model/plant_model.keras")



In [14]:
print(model.input_shape)

(None, 224, 224, 3)


In [15]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("cipherplant.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmp96nqn5mm\assets


INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmp96nqn5mm\assets


Saved artifact at 'C:\Users\admin\AppData\Local\Temp\tmp96nqn5mm'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2052074132432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074131088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074131664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074130320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074132816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074130704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074128976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074129936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074129744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052074132624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2052